JSON dataset
      ↓
Load dataset
      ↓
Train / validation / test split
      ↓
Load tokenizer
      ↓
Load Qwen model
      ↓
Inspect chat template
      ↓
Format conversations
      ↓
Tokenization
      ↓
Inspect labels / loss masking
      ↓
LoRA
      ↓
SFT
      ↓
Evaluation

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

True
Tesla T4


In [2]:
import json
from collections import Counter

with open("/content/electronics_tutoring_dataset.json", "r") as file:
    data = json.load(file)

print(len(data))
print("Kirchhoff law number:",sum(row["topic"] == "Kirchhoff's Laws" for row in data))
print("diodes number:",sum(row["topic"] == "Diodes" for row in data))
print("BJTs number:",sum(row["topic"] == "BJTs" for row in data))
print("MOSFETs number:",sum(row["topic"] == "MOSFETs" for row in data))
print("Op-amps number:",sum(row["topic"] == "Op-amps" for row in data))
print(" ")


required = ["id","topic","difficulty","instruction","question","answer"]

count = 0
for item in data:
  for field in required:
    if not item.get(field):
      count += 1

print(count)


250
Kirchhoff law number: 51
diodes number: 50
BJTs number: 50
MOSFETs number: 50
Op-amps number: 49
 
0


In [3]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(data,test_size=0.2,random_state=0,shuffle=True)

valid, test = train_test_split(temp,test_size=0.5,random_state=0,shuffle=True)


print("Train:", len(train))
print("Valid:", len(valid))
print("Test:", len(test))

Train: 200
Valid: 25
Test: 25


In [4]:
print(train[0])

{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer


checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [6]:
model.num_parameters()

1543714304

In [7]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(checkpoint)

In [8]:
print(config)

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [9]:
print("vocab size",len(tokenizer))
print("special tokens",tokenizer.special_tokens_map)
print("")
if tokenizer.chat_template:
  print(tokenizer.chat_template)

vocab size 151665
special tokens {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end

In [10]:

text = train[0]
print(text)
instruction = text["instruction"]
question = text["question"]
answer = text["answer"]


messages = [
    {
        "role" : "system",
        "content" : instruction
    },
    {
        "role" : "user",
        "content":question,
    },
    {
        "role" : "assistant",
        "content":answer
    }
]

to_chat_template = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt = False)
print(to_chat_template)
print("")

tokenized_message = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]
print(tokenized_message)

decoded_message = tokenizer.decode(tokenized_message)
print(decoded_message)



{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}
<|im_start|>system
Explain this concept as if you were tutoring an undergraduate engineering student.<|im_end|>
<|im_start|>user
Why does a diode conduct current in only one direction?<|im_end|>
<|im_start|>assistant
The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction.<|im_end|>


[151

In [11]:
print(tokenizer("<|im_start|>assistant",add_special_tokens=False)["input_ids"])
print(tokenizer("<|im_end|>",add_special_tokens=False)["input_ids"])

[151644, 77091]
[151645]


In [12]:
tokens = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]
for i,token in enumerate(tokens):
  if(tokens[i] == 151644 and tokens[i+1] == 77091):
    start = i+2

for j in range(start,len(tokens)):
  if(tokens[j] == 151645):
    end = j

print(start)
print(end)



38
83


In [13]:
labels = tokenizer(to_chat_template,add_special_tokens=False)["input_ids"]

for i in range(0,start):
  labels[i] = -100

for j in range(end,len(labels)):
  labels[j] = -100

print(len(tokens))
print(len(labels))
print((tokens[34:84]))
print((labels[34:84]))


85
85
[151645, 198, 151644, 77091, 198, 785, 61901, 48241, 594, 91848, 5537, 323, 5798, 3419, 9072, 2070, 33034, 1482, 6396, 304, 9931, 15470, 553, 22188, 8686, 18602, 7203, 11, 1393, 304, 4637, 15470, 279, 9251, 21720, 916, 6579, 419, 22103, 323, 6147, 8686, 34891, 311, 6396, 3941, 279, 48241, 13, 151645]
[-100, -100, -100, -100, 198, 785, 61901, 48241, 594, 91848, 5537, 323, 5798, 3419, 9072, 2070, 33034, 1482, 6396, 304, 9931, 15470, 553, 22188, 8686, 18602, 7203, 11, 1393, 304, 4637, 15470, 279, 9251, 21720, 916, 6579, 419, 22103, 323, 6147, 8686, 34891, 311, 6396, 3941, 279, 48241, 13, -100]


In [14]:
print(model.named_parameters)

<bound method Module.named_parameters of Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qw

In [15]:
!pip install -U torchao

In [16]:
from peft import get_peft_model,LoraConfig

peft_config = LoraConfig(
    target_modules = ["q_proj","v_proj"],
    task_type = "CAUSAL_LM",
    r = 8,
    lora_alpha = 16,
    lora_dropout= 0.05
)

peft_model = get_peft_model(model,peft_config)
peft_model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [17]:
print(train)

[{'id': 72, 'topic': 'Diodes', 'difficulty': 'intermediate', 'instruction': 'Explain this concept as if you were tutoring an undergraduate engineering student.', 'question': 'Why does a diode conduct current in only one direction?', 'answer': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}, {'id': 161, 'topic': 'MOSFETs', 'difficulty': 'basic', 'instruction': 'Give a clear conceptual explanation of the following electronics question.', 'question': 'What is the body effect in a MOSFET?', 'answer': 'The body effect is the change in threshold voltage that occurs when there is a voltage difference between the source and the body terminal, which effectively makes the threshold voltage a function of the source-body voltage rather than a fixed constant.'}, {'id': 181, 'topic': 'MO

In [18]:
messages_train = []

for text in train:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_train.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

In [19]:
print(len(messages_train))
print(messages_train[0])

200
[{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': 'Why does a diode conduct current in only one direction?'}, {'role': 'assistant', 'content': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}]


In [20]:
messages_validation = []

for text in valid:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_validation.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

In [21]:
print(len(messages_validation))
print(messages_validation[0])

25
[{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': "Why doesn't a MOSFET's gate draw significant current in steady state?"}, {'role': 'assistant', 'content': 'The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.'}]


In [22]:
from datasets import Dataset

# Convert each conversation into {"messages": [...]}
train_data = [{"messages": conversation} for conversation in messages_train]

train_dataset = Dataset.from_list(train_data)

print(len(train_dataset))
print(train_dataset[0])

200
{'messages': [{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': 'Why does a diode conduct current in only one direction?'}, {'role': 'assistant', 'content': "The PN junction's depletion region and built-in electric field oppose current flow in reverse bias by blocking majority carrier movement, while in forward bias the applied voltage overcomes this barrier and allows majority carriers to flow across the junction."}]}


In [23]:
valid_data = [{"messages": conversation} for conversation in messages_validation]

valid_dataset = Dataset.from_list(valid_data)

print(len(valid_dataset))
print(valid_dataset[0])

25
{'messages': [{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': "Why doesn't a MOSFET's gate draw significant current in steady state?"}, {'role': 'assistant', 'content': 'The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.'}]}


In [24]:
!pip install -U trl

KeyboardInterrupt: 

In [25]:
import trl
print(trl.__version__)

1.9.2


In [26]:
print(tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [27]:
from trl import SFTConfig,SFTTrainer

sft_config = SFTConfig(
    output_dir="./results",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    # Sequence length
    max_length=512,

    # Evaluation
    eval_strategy="epoch",

    # Saving
    save_strategy="epoch",

    # Train only on assistant responses
    assistant_only_loss=True,

    # Optional
    logging_steps=10,
    report_to="none",
)

trainer = SFTTrainer(model=model,args=sft_config,train_dataset=train_dataset,eval_dataset=valid_dataset)

trainer.train()

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.723506,1.672144,1.536437,18374.000000,0.576590
2,1.544584,1.581751,1.523387,36748.000000,0.598696
3,1.478145,1.556280,1.474564,55122.000000,0.609363


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=75, training_loss=1.6077486673990886, metrics={'train_runtime': 529.7025, 'train_samples_per_second': 1.133, 'train_steps_per_second': 0.142, 'total_flos': 486498617659392.0, 'train_loss': 1.6077486673990886, 'epoch': 3.0})

In [62]:
print(trainer.model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=1536, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=1536, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=1536, out_features=256, bias

In [60]:
trainer.save_model("./electronics-lora")
tokenizer.save_pretrained("./electronics-lora")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./electronics-lora/tokenizer_config.json',
 './electronics-lora/chat_template.jinja',
 './electronics-lora/tokenizer.json')

In [61]:
import os

print(os.listdir("./electronics-lora"))

['chat_template.jinja', 'model.safetensors', 'config.json', 'generation_config.json', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json']


Evaluation

A)Base Model (Qwen2.5-1.5B-Instruct)

B) Fine-tuned model (Qwen2.5-1.5B-Instruct + LoRA)

In [28]:
messages_test = []

for text in valid:
  instruction = text["instruction"]
  question = text["question"]
  answer = text["answer"]
  messages_test.append([
        {
            "role" : "system",
            "content" : instruction
        },
        {
            "role" : "user",
            "content":question,
        },
        {
            "role" : "assistant",
            "content":answer
        }
    ])

from datasets import Dataset

# Convert each conversation into {"messages": [...]}
test_data = [{"messages": conversation} for conversation in messages_test]

test_data = Dataset.from_list(test_data)

print(len(test_data))
print(test_data[0])

25
{'messages': [{'role': 'system', 'content': 'Explain this concept as if you were tutoring an undergraduate engineering student.'}, {'role': 'user', 'content': "Why doesn't a MOSFET's gate draw significant current in steady state?"}, {'role': 'assistant', 'content': 'The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.'}]}


In [43]:
test_sample1 = test_data[0]

instruction = test_sample1["messages"][0]["content"]
question = test_sample1["messages"][1]["content"]
reference_answer = test_sample1["messages"][2]["content"]

print("INSTRUCTION:")
print(instruction)

print("\nQUESTION:")
print(question)

print("\nREFERENCE ANSWER:")
print(reference_answer)

INSTRUCTION:
Explain this concept as if you were tutoring an undergraduate engineering student.

QUESTION:
Why doesn't a MOSFET's gate draw significant current in steady state?

REFERENCE ANSWER:
The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.


In [46]:
conversation = [
    {
        "role": "system",
        "content": instruction
    },
    {
        "role": "user",
        "content": question
    }
]

In [47]:
prompt = tokenizer.apply_chat_template(conversation,tokenize=False,add_generation_prompt=True)
print(prompt)

<|im_start|>system
Explain this concept as if you were tutoring an undergraduate engineering student.<|im_end|>
<|im_start|>user
Why doesn't a MOSFET's gate draw significant current in steady state?<|im_end|>
<|im_start|>assistant



In [48]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

generated = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("MODEL ANSWER:")
print(generated)

print("\nREFERENCE ANSWER:")
print(reference_answer)

MODEL ANSWER:
The drain-source voltage is typically much higher than the threshold voltage, so the gate-source voltage must be very close to zero for the device to turn on; it can only do that with negligible current flowing into the gate.

REFERENCE ANSWER:
The gate is separated from the channel by a thin insulating oxide layer, so no DC current can flow into the gate; the gate voltage instead controls the channel through an electric field.


In [49]:
checkpoint = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(checkpoint)
base_tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [50]:
inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False
)

generated = base_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("MODEL ANSWER:")
print(generated)

print("\nREFERENCE ANSWER:")
print(reference_answer)

MODEL ANSWER:
A MOSFET (Metal-Oxide-Semiconductor Field-Effect Transistor) is designed to control the flow of electrical current through its channel by applying voltage across it, rather than relying on direct current from a power source. This design allows for very high input impedance, which means that the gate terminal does not draw much current when the transistor is operating in its intended mode.

Here’s why:

1. **Input Impedance**: The primary function of a MOSFET is to act as a switch or amplifier with minimal resistance to the signal applied at its gate terminal. When the gate-to-source voltage \(V_{GS}\) is sufficiently large and positive relative to the threshold voltage, the MOSFET can conduct a substantial amount of current into the drain terminal. However, once the gate voltage drops below the threshold value, the device remains off until another gate pulse brings it back to conducting status.

2. **Threshold Voltage**: The threshold voltage (\(V_T\)) is the minimum gate

In [51]:
print(test[0])

{'id': 195, 'topic': 'MOSFETs', 'difficulty': 'applied', 'instruction': 'Explain this electronics concept to an undergraduate student.', 'question': "What happens to a MOSFET's drain current if the drain-source voltage is increased well beyond the overdrive voltage while gate-source voltage stays fixed?", 'answer': 'Once the transistor is in saturation, further increases in drain-source voltage have only a small effect on drain current (due to channel-length modulation), so the current stays approximately constant rather than continuing to rise proportionally.'}


In [54]:
import torch

def calculate_test_loss(model, test_data, tokenizer, max_length=512):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for sample in test_data:

        # Create the conversation WITHOUT the answer
        messages = [
            {
                "role": "system",
                "content": sample["instruction"]
            },
            {
                "role": "user",
                "content": sample["question"]
            }
        ]

        # Prompt ending at assistant generation
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Full conversation INCLUDING the correct answer
        full_messages = messages + [
            {
                "role": "assistant",
                "content": sample["answer"]
            }
        ]

        full_text = tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False
        )

        # Tokenize prompt and full conversation
        prompt_tokens = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        full_tokens = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        input_ids = full_tokens["input_ids"].to(model.device)
        attention_mask = full_tokens["attention_mask"].to(model.device)

        # Labels start as -100, meaning "don't calculate loss"
        labels = input_ids.clone()

        # Ignore everything belonging to system + user prompt
        prompt_length = prompt_tokens["input_ids"].shape[1]

        labels[:, :prompt_length] = -100

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

        # Count actual answer tokens
        num_answer_tokens = (labels != -100).sum().item()

        total_loss += outputs.loss.item() * num_answer_tokens
        total_tokens += num_answer_tokens

    return total_loss / total_tokens

In [57]:
fine_tuned_loss = calculate_test_loss(
    model,
    test,
    tokenizer
)

print("Fine-tuned model loss:", fine_tuned_loss)

Fine-tuned model loss: 1.6593436661546361


In [58]:
base_loss = calculate_test_loss(
    base_model,
    test,
    base_tokenizer
)

print("Base model loss:", base_loss)

Base model loss: 2.113218175221796


In [59]:
print(f"Base model loss:       {base_loss:.4f}")
print(f"Fine-tuned model loss: {fine_tuned_loss:.4f}")
print(f"Loss reduction:        {base_loss - fine_tuned_loss:.4f}")

Base model loss:       2.1132
Fine-tuned model loss: 1.6593
Loss reduction:        0.4539


Training:
3 epochs

Trainable parameters:
1,089,536 / 1,544,803,840
= 0.0705%

Validation loss:
1.5563

Test loss:
Base model       = 2.1132
Fine-tuned model = 1.6593

Test loss reduction:
21.5%